## &nbsp; Es wurde beschlossen, alle vier Varianten des Buttons zu testen:

**A:** Weißer Button „JETZT KAUFEN“

**B:** Roter Button „JETZT KAUFEN“

**C:** Weißer Button „ANGEBOTE ANSEHEN“

**D:** Roter Button „ANGEBOTE ANSEHEN“

In [1]:
# Import von Bibliotheken
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
from itertools import combinations

In [2]:
# Laden der Experimentdaten
# CSV-Dateien aus dem GitHub-Repository laden

base_url = "https://raw.githubusercontent.com/JuliaAltmann/Eniac_AB_testing/main/data/raw/"

eniac_a = pd.read_csv(base_url + "eniac_a.csv")
eniac_b = pd.read_csv(base_url + "eniac_b.csv")
eniac_c = pd.read_csv(base_url + "eniac_c.csv")
eniac_d = pd.read_csv(base_url + "eniac_d.csv")

In [3]:
eniac_a.head()

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,269,True,Homepage Version A - white SHOP NOW • http...
1,25,div,mySidebar,309,True,created 2021-09-14 • 14 days 0 hours 34 mi...
2,4,a,Mac,279,True,NaN
3,69,a,iPhone,246,True,NaN
4,105,a,Accessories,1235,True,NaN


In [4]:
eniac_b.head()

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,236,True,Homepage Version B - red SHOP NOW • https:...
1,25,div,mySidebar,304,True,created 2021-10-27 • 14 days 0 hours 34 mi...
2,4,a,Mac,268,True,NaN
3,69,a,iPhone,260,True,NaN
4,105,a,Accessories,1214,True,NaN


In [5]:
eniac_c.head()

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,288,True,Homepage Version C - white SEE DEALS • htt...
1,25,div,mySidebar,283,True,created 2021-10-27 • 14 days 0 hours 34 mi...
2,4,a,Mac,262,True,NaN
3,69,a,iPhone,234,True,NaN
4,105,a,Accessories,1288,True,NaN


In [6]:
eniac_d.head()

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,285,True,Homepage Version D - red SEE DEALS • https...
1,25,div,mySidebar,305,True,created 2021-10-27 • 14 days 0 hours 34 mi...
2,4,a,Mac,274,True,NaN
3,69,a,iPhone,243,True,NaN
4,105,a,Accessories,1267,True,NaN


## 1.&nbsp; Formulieren Sie die Nullhypothese und die Alternativhypothese.

**Null Hypothesis ($H_0$):** Alle Versionen haben die gleiche CTR.

**Alternative Hypothesis ($H_A$):** Es gibt einen Unterschied in der CTR zwischen den Versionen.


## 2.&nbsp; Wählen Sie ein geeignetes Signifikanzniveau alpha ($\alpha$) aus.

Dies entspricht einem Konfidenzniveau von 95 %. Das bedeutet, dass wir eine Wahrscheinlichkeit von 5 % für ein falsch-positives Ergebnis (Fehler erster Art) in Kauf nehmen.

In [7]:
alpha = 0.05

## 3.&nbsp; Wir erstellen eine zusammenfassende Datentabelle aus unseren Dateien.


In [8]:
# ermitteln die Anzahl der Klicks auf den zu testenden Button.

button_a = eniac_a.loc[eniac_a["Name"] == "SHOP NOW", "No. clicks"].sum()
button_b = eniac_b.loc[eniac_b["Name"] == "SHOP NOW", "No. clicks"].sum()
button_c = eniac_c.loc[eniac_c["Name"] == "SEE DEALS", "No. clicks"].sum()
button_d = eniac_d.loc[eniac_d["Name"] == "SEE DEALS", "No. clicks"].sum()

Click = [button_a, button_b, button_c, button_d]

Click

[np.int64(512), np.int64(281), np.int64(527), np.int64(193)]

In [9]:
# extrahieren die Visitors aus jeder Datei.

import re

Visitors = []

for df in [eniac_a, eniac_b, eniac_c, eniac_d]:

    snapshot = df.loc[
        df["Snapshot information"].str.contains(
            "visits", case=False, na=False
        ),
        "Snapshot information"
    ].iloc[0]

    visitors = int(re.search(r"(\d+)\s*visits", snapshot).group(1))

    Visitors.append(visitors)

Visitors

[25326, 24747, 24876, 25233]

In [10]:
# Wir erstellen eine Click-/No-click-Tabelle.

No_click = [
    Visitors[i] - Click[i]
    for i in range(4)
]

observed_results = pd.DataFrame(
    [Click, No_click],
    columns=["A", "B", "C", "D"],
    index=["Click", "No-click"]
)

observed_results

,A,B,C,D
Click,512,281,527,193
No-click,24814,24466,24349,25040


## 4.&nbsp; Wir berechnen die Testergebnisse mithilfe der Chi-Quadrat-Statistik.


### Berechnung der Freiheitsgrade


In [11]:
r = observed_results.shape[0]
r

2

In [12]:
n = observed_results.shape[1]
n

4

In [13]:
dof = (r - 1) * (n - 1)
dof

3

### Berechnung des Erwartungswerts für jede Zelle



In [14]:
# Zeilensummen berechnen – gespeichert als vertikaler DataFrame
row_totals = observed_results.sum(axis=1).to_frame()

# Spaltensummen berechnen – gespeichert als horizontales DataFrame
column_totals = observed_results.sum(axis=0).to_frame().T

# Calculate the grand total - a single value
grand_total = observed_results.values.sum()

# Berechne die Gesamtsumme – einen einzelnen Wert
proportions = column_totals/grand_total

# Multiplizieren Sie die Anteile mit den Zeilensummen, um die Erwartungswerte zu erhalten.
expected_results = row_totals.dot(proportions)

expected_results

,A,B,C,D
Click,382.486255,373.7419,375.690124,381.081721
No-click,24943.513745,24373.2581,24500.309876,24851.918279


### Berechnung der Chi-Quadrat-Statistik



In [15]:
chi_squared = (
    ((observed_results - expected_results) ** 2 / expected_results)
    .sum() # Die Spalten zusammenzählen
    .sum() # Spaltensummen addieren
)
chi_squared

np.float64(224.01877488058415)

## 5.&nbsp; Das Testergebnis interpretieren


In [16]:
critical_value = 7.815

Unser Chi-Quadrat-Wert beträgt 224,02 und ist damit viel größer als der kritische Wert von 7,815.

Deshalb lehnen wir die Nullhypothese ab.

Bei einem Signifikanzniveau von 5 % gibt es einen statistisch signifikanten Unterschied zwischen den vier Versionen.

Das bedeutet, dass die Änderungen an der Webseite einen Einfluss auf die Klickrate haben.

Wir können also sagen, dass nicht alle Versionen gleich gut funktionieren.

## 6.&nbsp; Zweite Lösungsvariante für die Aufgabe: die Programmierer-Methode

Verwendung von SciPy


In [17]:
chisq, pvalue, df, expected = chi2_contingency(observed_results)

In [18]:
chisq

np.float64(224.01877488058412)

In [19]:
pvalue

np.float64(2.7161216607868712e-48)

In [20]:
df

3

In [21]:
expected

array([[  382.48625502,   373.74189974,   375.69012397,   381.08172127],
       [24943.51374498, 24373.25810026, 24500.30987603, 24851.91827873]])

Überprüfung, ob die manuell berechnete Chi-Quadrat-Teststatistik mit derjenigen übereinstimmt, die mithilfe von scipy berechnet wurde.

In [22]:
round(chi_squared, 5) == round(chisq, 5)

np.True_

In [23]:
if pvalue > alpha:
    print("Der p-Wert ist größer als Alpha.")
else:
    print("Der p-Wert ist kleiner als Alpha.")

Der p-Wert ist kleiner als Alpha.


Der p-Wert beträgt 2,716 × 10⁻⁴⁸ und ist damit deutlich kleiner als das Signifikanzniveau von 0,05.

Deshalb lehnen wir die Nullhypothese ab.

Es gibt einen statistisch signifikanten Unterschied zwischen den vier Versionen.

## 7.&nbsp; Welcher Version ist besser?

In [24]:
# Berechne die Klickrate für jede Version.

ctr_a = observed_results.loc["Click", "A"] / sum(observed_results["A"])
ctr_b = observed_results.loc["Click", "B"] / sum(observed_results["B"])
ctr_c = observed_results.loc["Click", "C"] / sum(observed_results["C"])
ctr_d = observed_results.loc["Click", "D"] / sum(observed_results["D"])

# Erstelle eine Tabelle mit allen Versionen und deren Klickraten.
ctrs = pd.DataFrame(
    {"version": ["A", "B", "C", "D"], "CTR": [ctr_a, ctr_b, ctr_c, ctr_d]}
)
ctrs.sort_values("CTR", ascending=False)

,version,CTR
2,C,0.021185
0,A,0.020216
1,B,0.011355
3,D,0.007649


Version C hat mit 2,12 % die höchste Klickrate und ist damit die beste Version.

Version A liegt mit 2,02 % knapp dahinter.

Version B und D haben deutlich niedrigere Klickraten.

Deshalb ist Version C unser Favorit.

## 8.&nbsp; Überprüfung, ob Version C statistisch gesehen tatsächlich besser ist als die anderen


Es sind 6 paarweise Vergleiche von vier Versionen durchzuführen:
- A vs. B
- A vs. C
- A vs. D
- B vs. C
- B vs. D
- C vs. D

Bei der Durchführung von insgesamt 6 Tests müssen multiple Vergleiche berücksichtigt werden. Hierfür verwenden wir die Bonferroni-Korrektur.

Bei einem ursprünglichen Signifikanzniveau von alpha = 0,05:

$$
\alpha_{adjusted} = \frac{0.05}{6} = 0.00833
$$

Das bedeutet, dass der p-Wert jedes einzelnen Vergleichs nun unter 0,00833 liegen muss.

#### Wir führen paarweise Chi-Quadrat-Tests durch.

In [25]:
# Wir gehen alle möglichen Paare von Versionen durch.
versions = ["A", "B", "C", "D"]
pairs = list(combinations(versions, 2))

results = []

for version1, version2 in pairs:

    # erstellen eine Tabelle für die beiden zu vergleichenden Versionen.
    table = observed_results[[version1, version2]]

    # Wir führen einen Chi-Quadrat-Test durch.
    chi_squared, p_value, dof, expected = chi2_contingency(
        table,
        correction=False
    )

    results.append({
        "Comparison": f"{version1} vs {version2}",
        "Chi-square": chi_squared,
        "P-value": p_value
    })

# erstellen eine Ergebnistabelle.
pairwise_results = pd.DataFrame(results)

# Bonferroni-Korrektur
pairwise_results["Adjusted p-value"] = (
    pairwise_results["P-value"] * 6
).clip(upper=1)

# prüfen die statistische Signifikanz.
pairwise_results["Significant"] = (
    pairwise_results["Adjusted p-value"] < 0.05
)

pairwise_results

,Comparison,Chi-square,P-value,Adjusted p-value,Significant
0,A vs B,63.060892,2.004141e-15,1.202485e-14,True
1,A vs C,0.581022,4.459121e-01,1.000000e+00,False
2,A vs D,145.195779,1.946072e-33,1.167643e-32,True
3,B vs C,74.841623,5.100301e-18,3.060181e-17,True
4,B vs D,18.268744,1.918290e-05,1.150974e-04,True
5,C vs D,162.071440,3.990996e-37,2.394597e-36,True


Was bedeutet das für:

**A:** Weißer Button „JETZT KAUFEN“

**B:** Roter Button „JETZT KAUFEN“

**C:** Weißer Button „ANGEBOTE ANSEHEN“

**D:** Roter Button „ANGEBOTE ANSEHEN“?

# Zentrale Ergebnisse
- Version C erzielt mit 2,12 % die höchste CTR.
- Version A erreicht mit 2,02 % den zweithöchsten Wert.
- Zwischen A und C besteht kein statistisch signifikanter Unterschied.
- C schneidet statistisch signifikant besser ab als B und D.
- Auch A schneidet statistisch signifikant besser ab als B und D.

#Fazit

Version C weist die höchste beobachtete Klickrate auf. Allerdings besteht kein statistisch signifikanter Unterschied zwischen Version A und Version C.

Beide Varianten schneiden signifikant besser ab als B und D. Daher können A und C als die leistungsstärksten Varianten betrachtet werden. Ein eindeutiger statistischer Gewinner lässt sich jedoch nicht bestimmen.